In [0]:
# Databricks notebook: 02_transform_silver
# Cleans Bronze tables and writes to Silver (schema: 02_silver)

from pyspark.sql.functions import col, when, current_timestamp
from pyspark.sql.types import DecimalType, ByteType

catalog_name = "retail_catalog"
bronze_schema = f"{catalog_name}.01_bronze"
silver_schema = f"{catalog_name}.02_silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

decimal_cols_high = {
    "unitprice", "unitcost", "grossvalue", "discountamount",
    "taxamount", "shipcost", "annualrentcost", "budget", "maxdiscountcap"
}
decimal_cols_pct = {
    "margin_pct", "discount_pct", "redemption_rate",
    "promoupliftfactor", "tax_rate", "storesizemultiplier", "spendmultiplier"
}

pk_mapping = {
    "dimdate":      "datekey",
    "dimcustomer":  "customerid",
    "dimproduct":   "productid",
    "dimstore":     "storeid",
    "dimpromotion": "promoid",
    "factsales":    "salesid"
}

for tbl in spark.sql(f"SHOW TABLES IN {bronze_schema}").collect():
    bronze_name = tbl.tableName
    entity = bronze_name.replace("bronze_", "")
    silver_name = f"silver_{entity}"
    print(f"Processing {bronze_name} → {silver_name}...")

    df = spark.table(f"{bronze_schema}.{bronze_name}")

    select_exprs = []
    for c in df.columns:
        expr = col(c)
        if c in decimal_cols_high:
            expr = expr.cast(DecimalType(18, 2))
        elif c in decimal_cols_pct:
            if c in ("margin_pct", "discount_pct"):
                expr = expr.cast(DecimalType(5, 4))
            else:
                expr = expr.cast(DecimalType(10, 4))
        elif c == "hour":
            expr = expr.cast(ByteType())
        # Negate qty for returns (only for factsales)
        if c == "qty" and "factsales" in bronze_name:
            expr = when((col("isreturn") == 1) & (col("qty") > 0), -col("qty")).otherwise(col("qty"))
        if df.schema[c].dataType.typeName() == "string":
            expr = when(expr == "", None).otherwise(expr)
        select_exprs.append(expr.alias(c))

    df = df.select(*select_exprs)

    if entity in pk_mapping:
        df = df.dropDuplicates([pk_mapping[entity]])
    else:
        df = df.dropDuplicates()

    df = df.withColumn("_silver_processed_ts", current_timestamp())
    df = df.repartition(4)

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{silver_name}")
    print(f"✓ Created {silver_schema}.{silver_name} with {df.count():,} rows\n")

# Dummy promotion row
spark.sql(f"DELETE FROM {silver_schema}.silver_dimpromotion WHERE promoid = 0")
spark.sql(f"""
    INSERT INTO {silver_schema}.silver_dimpromotion
    (promoid, promoname, discount_pct, discount_fixed, type, isactive, minspend, channel,
     budget, startdate, enddate, targetaudience, maxdiscountcap, isstackable,
     redemption_rate, coderequired, promoupliftfactor)
    VALUES (0, 'No Promotion', 0.0000, 0.00, 'None', 1, 0, 'All', 0.00,
            '2000-01-01', '2099-12-31', 'All', 0.00, 0, 0.000, 0, 1.000)
""")
print("Dummy promotion row ensured.")

print("Silver layer transformation completed successfully.")

Processing bronze_dimcustomer → silver_dimcustomer...
✓ Created retail_catalog.02_silver.silver_dimcustomer with 200,001 rows

Processing bronze_dimdate → silver_dimdate...
✓ Created retail_catalog.02_silver.silver_dimdate with 1,243 rows

Processing bronze_dimproduct → silver_dimproduct...
✓ Created retail_catalog.02_silver.silver_dimproduct with 2,001 rows

Processing bronze_dimpromotion → silver_dimpromotion...
✓ Created retail_catalog.02_silver.silver_dimpromotion with 102 rows

Processing bronze_dimstore → silver_dimstore...
✓ Created retail_catalog.02_silver.silver_dimstore with 201 rows

Processing bronze_factsales → silver_factsales...
✓ Created retail_catalog.02_silver.silver_factsales with 10,000,000 rows

Dummy promotion row ensured.
Silver layer transformation completed successfully.
